In [1]:
# pip install torchmetrics
%pip install -q dagshub mlflow torchmetrics focal-loss-torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.7/24.7 MB 58.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 49.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.0 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.3 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import dagshub
dagshub.init(repo_owner='pratham.doshi', repo_name='faster_rcnn_with_mlflow', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=31678e0e-de45-4c14-be6a-85e82f905cbb&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=178628f29a6c3db4d9a8e724129e890186db638b41df67c34bb88c0c0e352c2d




Output()

Accessing as pratham.doshi

Initialized MLflow to track repo "pratham.doshi/faster_rcnn_with_mlflow"

Repository pratham.doshi/faster_rcnn_with_mlflow initialized!

In [17]:
import os
import xml.etree.ElementTree as ET
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

from torchvision import transforms as T
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import box_convert

from torchmetrics.detection.mean_ap import MeanAveragePrecision

from focal_loss import FocalLoss

import mlflow
import mlflow.pytorch

import optuna
from tqdm import tqdm


In [5]:
class YoloDataset(Dataset):
    def __init__(self, img_dir, label_dir, transforms=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transforms = transforms
        self.images = sorted([f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.jpeg', '.png'))])
        print(f"Found {len(self.images)} images")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert("RGB")
        width, height = image.size

        label_file = img_name.rsplit('.', 1)[0] + '.txt'
        label_path = os.path.join(self.label_dir, label_file)

        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    if line.strip() == "":
                        continue
                    class_id, x_center, y_center, w, h = map(float, line.strip().split())

                    # Convert YOLO normalized format to absolute [xmin, ymin, xmax, ymax]
                    x_center *= width
                    y_center *= height
                    w *= width
                    h *= height

                    xmin = x_center - w / 2
                    ymin = y_center - h / 2
                    xmax = x_center + w / 2
                    ymax = y_center + h / 2

                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(int(class_id) + 1)  # Faster R-CNN expects class labels starting from 1

        if len(boxes) == 0:
            return None  # You can also handle skipping None in DataLoader with custom collate_fn

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': torch.tensor([idx]),  # required for evaluation
            'area': (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]),
            'iscrowd': torch.zeros((len(boxes),), dtype=torch.int64),  # assuming all instances are not crowd
        }

        if self.transforms:
            image = self.transforms(image)

        return image, target

transform = T.ToTensor()

In [6]:
def collate_fn(batch):
    batch = list(filter(lambda x: x is not None, batch))  # remove None entries
    return tuple(zip(*batch))

train_dataset = YoloDataset(
    # img_dir='/content/drive/MyDrive/PPE Detection.v1i.yolov8/train/images',
    # /kaggle/input/ppe-data/train/images
    img_dir="/kaggle/input/ppe-data/train/images",
    label_dir='/kaggle/input/ppe-data/train/labels',
    transforms=transform
)

valid_dataset = YoloDataset(
    img_dir='/kaggle/input/ppe-data/valid/images',
    label_dir='/kaggle/input/ppe-data/valid/labels',
    transforms=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)

Found 7133 images
Found 311 images


In [7]:
test_dataset = YoloDataset(
    img_dir='/kaggle/input/ppe-data/test/images',
    label_dir='/kaggle/input/ppe-data/test/labels',
    transforms=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)


Found 139 images


In [8]:
from torchvision.models.detection.roi_heads import RoIHeads
import torch.nn.functional as F

class FocalRoIHeads(RoIHeads):
    def __init__(self, *args, focal_loss=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = focal_loss

    def fastrcnn_loss(self, class_logits, box_regression, labels, regression_targets):
        N, num_classes = class_logits.shape
        class_logits = class_logits.reshape(-1, num_classes)
        labels = torch.cat(labels, dim=0)

        # Apply softmax before passing to focal loss
        class_probs = F.softmax(class_logits, dim=-1)

        # Use your focal loss (assumes it handles ignore_index correctly)
        classification_loss = self.focal_loss(class_probs, labels)

        # Bounding box loss (unchanged)
        sampled_pos_inds_subset = torch.where(labels > 0)[0]
        labels_pos = labels[sampled_pos_inds_subset]
        box_regression = box_regression.reshape(N, -1, 4)

        box_loss = F.smooth_l1_loss(
            box_regression[sampled_pos_inds_subset, labels_pos],
            regression_targets[sampled_pos_inds_subset],
            beta=1.0,
            reduction="sum"
        ) / labels.numel()

        return classification_loss, box_loss


In [9]:
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone

def get_model_with_focal_loss(num_classes, focal_loss):
    backbone = resnet_fpn_backbone('resnet50', pretrained=True)

    dummy_model = FasterRCNN(backbone, num_classes=num_classes)

    roi_heads = FocalRoIHeads(
        box_roi_pool=dummy_model.roi_heads.box_roi_pool,
        box_head=dummy_model.roi_heads.box_head,
        box_predictor=dummy_model.roi_heads.box_predictor,
        fg_iou_thresh=0.5,
        bg_iou_thresh=0.5,
        batch_size_per_image=512,
        positive_fraction=0.25,
        bbox_reg_weights=None,
        score_thresh=0.05,
        nms_thresh=0.5,
        detections_per_img=100,
        focal_loss=focal_loss,
    )

    model = FasterRCNN(
        backbone=backbone,
        num_classes=num_classes,
        rpn_anchor_generator=dummy_model.rpn.anchor_generator,
        box_roi_pool=dummy_model.roi_heads.box_roi_pool,
        roi_heads=roi_heads
    )

    return model


In [11]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Memory Allocated: {torch.cuda.memory_allocated(0)/(1024**2):.2f} MB")

Using device: cuda:0
GPU Name: Tesla P100-PCIE-16GB
Memory Allocated: 0.00 MB


In [12]:
from tqdm import tqdm

def train_one_epoch(model, optimizer, data_loader, device, epoch=0):
    model.train()
    total_loss = 0.0

    progress_bar = tqdm(data_loader, desc=f" Epoch {epoch+1}", leave=True)

    for batch_idx, (images, targets) in enumerate(progress_bar):
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        batch_loss = losses.item()
        total_loss += batch_loss

        # Optional: extract individual loss types (if needed)
        cls_loss = loss_dict.get("loss_classifier", torch.tensor(0)).item()
        box_loss = loss_dict.get("loss_box_reg", torch.tensor(0)).item()
        rpn_cls_loss = loss_dict.get("loss_objectness", torch.tensor(0)).item()
        rpn_box_loss = loss_dict.get("loss_rpn_box_reg", torch.tensor(0)).item()

        avg_loss = total_loss / (batch_idx + 1)
        progress_bar.set_postfix({
            "Batch Loss": f"{batch_loss:.4f}",
            "Avg Loss": f"{avg_loss:.4f}",
            "Cls": f"{cls_loss:.4f}",
            "Box": f"{box_loss:.4f}"
        })

    return total_loss / len(data_loader)


In [20]:
num_classes = 5

In [21]:
def objective(trial):
    epochs = trial.suggest_int("epochs", 10, 20, step=5)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [2, 4, 8])
    optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    gamma = trial.suggest_float("focal_gamma", 1.0, 3.0)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

    focal_loss = FocalLoss(gamma=gamma, reduction='mean', ignore_index=-100)
    model = get_model_with_focal_loss(num_classes=num_classes, focal_loss=focal_loss).to(device)

    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9, weight_decay=weight_decay)
    else:
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    best_map50 = 0.0

    # Start MLflow logging
    with mlflow.start_run(nested=True):
        mlflow.log_params({
            "epochs": epochs,
            "learning_rate": learning_rate,
            "batch_size": batch_size,
            "optimizer": optimizer_name,
            "weight_decay": weight_decay,
            "focal_gamma": gamma,
        })

        for epoch in range(epochs):
            epoch_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)

            # Evaluation
            model.eval()
            metric = MeanAveragePrecision()
            with torch.no_grad():
                for images, targets in valid_loader:
                    images = [img.to(device) for img in images]
                    targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

                    outputs = model(images)
                    preds = []
                    gts = []
                    for pred, gt in zip(outputs, targets):
                        preds.append({
                            "boxes": pred["boxes"].cpu(),
                            "scores": pred["scores"].cpu(),
                            "labels": pred["labels"].cpu(),
                        })
                        gts.append({
                            "boxes": gt["boxes"].cpu(),
                            "labels": gt["labels"].cpu(),
                        })
                    metric.update(preds, gts)

            results = metric.compute()
            map50 = results["map_50"].item()
            average_map = results["map"].item
            mlflow.log_metric("map_50", map50, step=epoch)
            mlflow.log_metric("map", average_map, step=epoch)
            

            trial.report(map50, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

            # Save best model
            if map50 > best_map50:
                best_map50 = map50
                model_path = f"best_model_trial_{trial.number}.pth"
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': epoch_loss,
                }, model_path)
                mlflow.log_artifact(model_path)
                os.remove(model_path)

        mlflow.log_metric("best_map_50", best_map50)

    return best_map50


In [ ]:
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_startup_trials=5))
study.optimize(objective, n_trials=20)

print("Best trial:", study.best_trial)


[I 2025-07-07 09:54:43,871] A new study created in memory with name: no-name-eb8e8508-c3b3-410f-99e1-e58019e29465
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:135: UserWarning: Using 'backbone_name' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloadin

In [ ]:
A